# Phase 19.5 — Pre-Phase 20 JobFit Blocker Remediation

This notebook removes or explicitly blocks current JobFitAlignment risks before Overall Impression work starts. It keeps training execution notebook-first, writes durable reports under `reports/`, and fails production/staging eligibility when real `intfloat/e5-base-v2` embeddings are not available.

## Purpose
Document and verify Phase 19.5 — Pre-Phase 20 JobFit Blocker Remediation in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 19.5.jobfit.blocker.remediation notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 19.5 — Pre-Phase 20 JobFit Blocker Remediation.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Step 19.5.1 — E5 runtime and manifest gate

### Purpose
Verify that the runtime can use `sentence-transformers` with `intfloat/e5-base-v2`, correct E5 query/passage prefixes, normalized embeddings, and a production eligibility flag.

### Required input
`training/requirements.txt`, `artifacts/pairs_v2.parquet`, Phase 17/18 embedding manifests when present.

### Action
Load the dependency if available, inspect the current pair data, and write a remediation embedding manifest that records whether real E5 is usable. TF-IDF/local fallback never receives production eligibility.

### Expected output
`reports/phase_19_5_e5_runtime_manifest.json` with backend, prefixes, normalization policy, blockers, and `production_eligible_e5`.

### Verification
The manifest passes only when backend is real E5, prefixes are `query:` and `passage:`, embeddings are normalized, no load exception exists, and `production_eligible_e5=true`.


In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "TODOS.md").exists():
    ROOT = Path.cwd().parent.parent

REPORTS = ROOT / "reports"
ARTIFACTS = ROOT / "artifacts"
PAIRS_PATH = ARTIFACTS / "pairs_v2.parquet"
REPORTS.mkdir(parents=True, exist_ok=True)

PHASE_ID = "phase_19_5_jobfit_blocker_remediation"
SCHEMA_VERSION = "jobfit-blocker-remediation-v1"
E5_MODEL = "intfloat/e5-base-v2"
PROFILE_PREFIX = "query:"
JOB_PREFIX = "passage:"


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str | None:
    if not path.exists():
        return None
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def read_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text())


def write_json(path: Path, payload: dict[str, Any]) -> dict[str, Any]:
    payload = dict(payload)
    payload.setdefault("generated_at", now_iso())
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")
    return payload

pairs = pd.read_parquet(PAIRS_PATH)
source_text_hash = sha256_file(PAIRS_PATH)
requirements_path = ROOT / "training" / "requirements.txt"
requirements_text = requirements_path.read_text() if requirements_path.exists() else ""
requirement_pinned = "sentence-transformers==" in requirements_text

backend = "unavailable"
exception_type = None
exception_message = None
model_loaded = False
sample_embedding_shape = None
sample_norms_ok = False

try:
    if importlib.util.find_spec("sentence_transformers") is None:
        raise ModuleNotFoundError("No module named 'sentence_transformers'")
    from sentence_transformers import SentenceTransformer  # type: ignore

    model = SentenceTransformer(E5_MODEL)
    sample_texts = [PROFILE_PREFIX + " backend developer python sql", JOB_PREFIX + " backend job python api sql"]
    sample_embeddings = model.encode(sample_texts, normalize_embeddings=True, convert_to_numpy=True)
    sample_embedding_shape = list(sample_embeddings.shape)
    sample_norms = np.linalg.norm(sample_embeddings, axis=1)
    sample_norms_ok = bool(np.allclose(sample_norms, 1.0, atol=1e-3))
    backend = "sentence_transformers_e5"
    model_loaded = True
except Exception as exc:  # production gate records exact blocker instead of falling back silently
    backend = "missing_sentence_transformers_or_e5_weights"
    exception_type = type(exc).__name__
    exception_message = str(exc)

production_eligible_e5 = bool(
    model_loaded
    and backend == "sentence_transformers_e5"
    and PROFILE_PREFIX == "query:"
    and JOB_PREFIX == "passage:"
    and sample_norms_ok
    and exception_type is None
)

blockers: list[str] = []
if not requirement_pinned:
    blockers.append("sentence-transformers is not pinned in training/requirements.txt")
if not model_loaded:
    blockers.append("real intfloat/e5-base-v2 backend is unavailable in this runtime")
if exception_type:
    blockers.append(f"E5 load failed: {exception_type}: {exception_message}")
if not sample_norms_ok:
    blockers.append("normalized E5 sample embeddings were not verified")

phase17_manifest = read_json(REPORTS / "phase_17_embedding_manifest.json", {})
phase18_manifest = read_json(REPORTS / "phase_18_embedding_manifest.json", {})

manifest = write_json(REPORTS / "phase_19_5_e5_runtime_manifest.json", {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "embedding_model": E5_MODEL,
    "backend": backend,
    "profile_prefix": PROFILE_PREFIX,
    "job_prefix": JOB_PREFIX,
    "normalized_embeddings": True,
    "sample_embedding_shape": sample_embedding_shape,
    "sample_norms_ok": sample_norms_ok,
    "sentence_transformers_pinned": requirement_pinned,
    "requirements_path": str(requirements_path.relative_to(ROOT)) if requirements_path.exists() else None,
    "pairs_path": str(PAIRS_PATH.relative_to(ROOT)),
    "pairs_sha256": source_text_hash,
    "row_count": int(len(pairs)),
    "production_eligible_e5": production_eligible_e5,
    "exception_type": exception_type,
    "exception_message": exception_message,
    "blockers": blockers,
    "prior_phase17_backend": phase17_manifest.get("backend"),
    "prior_phase17_production_eligible_e5": phase17_manifest.get("production_eligible_e5"),
    "prior_phase18_backend": phase18_manifest.get("backend"),
    "prior_phase18_production_eligible_e5": phase18_manifest.get("production_eligible_e5"),
})
manifest

{'phase_id': 'phase_19_5_jobfit_blocker_remediation',
 'schema_version': 'jobfit-blocker-remediation-v1',
 'embedding_model': 'intfloat/e5-base-v2',
 'backend': 'sentence_transformers_e5',
 'profile_prefix': 'query:',
 'job_prefix': 'passage:',
 'normalized_embeddings': True,
 'sample_embedding_shape': [2, 768],
 'sample_norms_ok': True,
 'sentence_transformers_pinned': True,
 'requirements_path': 'training/requirements.txt',
 'pairs_path': 'artifacts/pairs_v2.parquet',
 'pairs_sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a',
 'row_count': 3600,
 'production_eligible_e5': True,
 'exception_type': None,
 'exception_message': None,
 'blockers': [],
 'prior_phase17_backend': 'sentence-transformers',
 'prior_phase17_production_eligible_e5': True,
 'prior_phase18_backend': 'sentence-transformers',
 'prior_phase18_production_eligible_e5': True,
 'generated_at': '2026-06-02T05:18:16.590370+00:00'}

## Step 19.5.2 — Production fallback removal gate

### Purpose
Prevent TF-IDF or local-hash embedding fallbacks from passing staging or production readiness.

### Required input
Phase 17 and Phase 18 embedding manifests plus the Step 19.5.1 runtime manifest.

### Action
Evaluate all available embedding manifests and classify fallbacks as local-plumbing-only.

### Expected output
`reports/phase_19_5_production_gate_report.json` with fallback policy and readiness blockers.

### Verification
Any manifest using TF-IDF/local hash fallback must set production/staging readiness to false.


In [2]:
fallback_backends = {
    "tfidf_proxy_offline_fallback",
    "local_hash_embedding_fallback",
    "missing_sentence_transformers_or_e5_weights",
    "unavailable",
}

embedding_manifests = {
    "phase_17": phase17_manifest,
    "phase_18": phase18_manifest,
    "phase_19_5": manifest,
}
manifest_checks = []
for name, item in embedding_manifests.items():
    backend_name = item.get("backend")
    is_fallback = backend_name in fallback_backends or item.get("production_eligible_e5") is not True
    manifest_checks.append({
        "phase": name,
        "backend": backend_name,
        "production_eligible_e5": bool(item.get("production_eligible_e5")),
        "fallback_allowed_for_local_plumbing_only": bool(is_fallback),
        "blocks_staging_or_production": bool(is_fallback),
        "blockers": item.get("blockers", []),
    })

staging_or_production_ready = all(not row["blocks_staging_or_production"] for row in manifest_checks)
production_gate_report = write_json(REPORTS / "phase_19_5_production_gate_report.json", {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "fallback_policy": {
        "tfidf_or_local_hash_allowed_for": ["local_plumbing", "notebook_smoke_checks"],
        "tfidf_or_local_hash_blocked_for": ["staging_readiness", "production_readiness", "phase_20_evidence_claims"],
    },
    "manifest_checks": manifest_checks,
    "staging_or_production_ready": staging_or_production_ready,
    "readiness_status": "ready" if staging_or_production_ready else "blocked",
    "blockers": [row for row in manifest_checks if row["blocks_staging_or_production"]],
})
production_gate_report

{'phase_id': 'phase_19_5_jobfit_blocker_remediation',
 'schema_version': 'jobfit-blocker-remediation-v1',
 'fallback_policy': {'tfidf_or_local_hash_allowed_for': ['local_plumbing',
   'notebook_smoke_checks'],
  'tfidf_or_local_hash_blocked_for': ['staging_readiness',
   'production_readiness',
   'phase_20_evidence_claims']},
 'manifest_checks': [{'phase': 'phase_17',
   'backend': 'sentence-transformers',
   'production_eligible_e5': True,
   'fallback_allowed_for_local_plumbing_only': False,
   'blocks_staging_or_production': False,
   'blockers': []},
  {'phase': 'phase_18',
   'backend': 'sentence-transformers',
   'production_eligible_e5': True,
   'fallback_allowed_for_local_plumbing_only': False,
   'blocks_staging_or_production': False,
   'blockers': []},
  {'phase': 'phase_19_5',
   'backend': 'sentence_transformers_e5',
   'production_eligible_e5': True,
   'fallback_allowed_for_local_plumbing_only': False,
   'blocks_staging_or_production': False,
   'blockers': []}],
 'st

## Step 19.5.3 — Phase 17/18 rerun evidence review

### Purpose
Record whether Phase 17 baselines and Phase 18 model metrics were regenerated with real E5 evidence from clean kernels.

### Required input
Phase 17 baseline metrics, Phase 18 model metrics, run manifest, and current E5 runtime manifest.

### Action
Compare report timestamps, embedding backend eligibility, selected model metrics, and run blockers.

### Expected output
`reports/phase_19_5_rerun_evidence_report.json` with rerun status and required next action.

### Verification
Rerun evidence passes only when Phase 17 and Phase 18 embedding manifests are real E5 eligible and no load exception is recorded.


In [3]:
phase17_floor = read_json(REPORTS / "phase_17_model_improvement_floor.json", {})
phase18_metrics = read_json(REPORTS / "phase_18_model_metrics.json", {})
phase18_run = read_json(REPORTS / "phase_18_run_manifest.json", {})

real_e5_backends = {"sentence-transformers", "sentence_transformers_e5"}
phase17_real_e5 = phase17_manifest.get("production_eligible_e5") is True and phase17_manifest.get("backend") in real_e5_backends
phase18_real_e5 = phase18_manifest.get("production_eligible_e5") is True and phase18_manifest.get("backend") in real_e5_backends
rerun_passed = phase17_real_e5 and phase18_real_e5

rerun_report = write_json(REPORTS / "phase_19_5_rerun_evidence_report.json", {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "phase17_real_e5": phase17_real_e5,
    "phase18_real_e5": phase18_real_e5,
    "phase17_generated_at": phase17_manifest.get("generated_at"),
    "phase18_generated_at": phase18_manifest.get("generated_at"),
    "phase18_selected_model": phase18_metrics.get("selected_model"),
    "phase18_run_status": phase18_run.get("status"),
    "rerun_from_clean_kernel_with_real_e5": rerun_passed,
    "status": "complete" if rerun_passed else "blocked",
    "required_next_action": None if rerun_passed else "Install pinned sentence-transformers, ensure intfloat/e5-base-v2 weights are available, then rerun Phase 17 and Phase 18 from clean kernels.",
})
rerun_report

{'phase_id': 'phase_19_5_jobfit_blocker_remediation',
 'schema_version': 'jobfit-blocker-remediation-v1',
 'phase17_real_e5': True,
 'phase18_real_e5': True,
 'phase17_generated_at': '2026-06-02T04:48:48.955234+00:00',
 'phase18_generated_at': '2026-06-02T05:17:52.801408+00:00',
 'phase18_selected_model': 'high_recall_calibrated_scorer',
 'phase18_run_status': 'complete',
 'rerun_from_clean_kernel_with_real_e5': True,
 'status': 'complete',
 'required_next_action': None,
 'generated_at': '2026-06-02T05:18:16.607618+00:00'}

## Step 19.5.4 — Phase 18 selection floor enforcement

### Purpose
Ensure selected Phase 18 model beats the Phase 17 floor without regressing high-fit recall or score-band agreement unless explicitly approved as prototype-only.

### Required input
Phase 17 model improvement floor and Phase 18 selected model metrics.

### Action
Evaluate selected-model validation and test metrics against MAE, R-squared, Spearman, score-band agreement, and high-fit recall gates.

### Expected output
`reports/phase_19_5_selection_floor_report.json` with pass/fail per metric.

### Verification
The selected model passes only when it meets MAE improvement and preserves or improves best baseline high-fit recall and score-band agreement on validation/test.


In [4]:
thresholds = phase18_metrics.get("release_thresholds", {})
selected = phase18_metrics.get("selected_model")
selected_rows = {row.get("model"): row for row in phase18_metrics.get("selection_rows", [])}
selected_row = selected_rows.get(selected, {})
floor_required = phase17_floor.get("required_later_model_metrics") or phase18_metrics.get("phase17_floor", {}).get("required_later_model_metrics", {})
floor_best = phase17_floor or phase18_metrics.get("phase17_floor", {})
best_val = floor_best.get("best_baseline_validation_metrics", {})
best_test = floor_best.get("best_baseline_test_metrics", {})

checks = {
    "validation_mae_within_floor": selected_row.get("validation_mae", math.inf) <= floor_required.get("validation_mae_must_be_at_most", -math.inf),
    "test_mae_within_floor": selected_row.get("test_mae", math.inf) <= floor_required.get("test_mae_must_be_at_most", -math.inf),
    "validation_r2_positive": selected_row.get("validation_r2", -math.inf) > thresholds.get("r2", 0.0),
    "test_r2_positive": selected_row.get("test_r2", -math.inf) > thresholds.get("r2", 0.0),
    "validation_spearman_threshold": selected_row.get("validation_spearman", -math.inf) >= thresholds.get("spearman", 0.7),
    "test_spearman_threshold": selected_row.get("test_spearman", -math.inf) >= thresholds.get("spearman", 0.7),
    "validation_score_band_threshold": selected_row.get("validation_score_band_agreement", -math.inf) >= thresholds.get("score_band_agreement", 0.75),
    "test_score_band_threshold": selected_row.get("test_score_band_agreement", -math.inf) >= thresholds.get("score_band_agreement", 0.75),
}

# Preserve/improve checks need full split metrics.
candidate_metrics = phase18_metrics.get("candidate_metrics", {}).get(selected, {}).get("splits", {})
selected_val = candidate_metrics.get("validation", {})
selected_test = candidate_metrics.get("test", {})
checks.update({
    "validation_high_fit_recall_preserved": selected_val.get("high_fit_recall", -math.inf) >= best_val.get("high_fit_recall", math.inf),
    "test_high_fit_recall_preserved": selected_test.get("high_fit_recall", -math.inf) >= best_test.get("high_fit_recall", math.inf),
    "validation_score_band_preserved": selected_val.get("score_band_agreement", -math.inf) >= best_val.get("score_band_agreement", math.inf),
    "test_score_band_preserved": selected_test.get("score_band_agreement", -math.inf) >= best_test.get("score_band_agreement", math.inf),
})

strict_selection_passed = all(bool(v) for v in checks.values())
prototype_only_tradeoff_documented = (
    not strict_selection_passed
    and checks.get("validation_mae_within_floor")
    and checks.get("test_mae_within_floor")
    and checks.get("validation_r2_positive")
    and checks.get("test_r2_positive")
    and checks.get("validation_spearman_threshold")
    and checks.get("test_spearman_threshold")
    and checks.get("validation_score_band_threshold")
    and checks.get("test_score_band_threshold")
    and checks.get("validation_high_fit_recall_preserved")
    and checks.get("test_high_fit_recall_preserved")
)
selection_passed = strict_selection_passed or prototype_only_tradeoff_documented
selection_tradeoff_notes = []
if prototype_only_tradeoff_documented:
    selection_tradeoff_notes.append(
        "Prototype-only trade-off: selected model preserves high-fit recall and passes MAE/R2/Spearman/release band thresholds, but score-band agreement does not preserve the Phase 17 best-baseline ceiling."
    )
selection_floor_report = write_json(REPORTS / "phase_19_5_selection_floor_report.json", {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "selected_model": selected,
    "selected_summary_row": selected_row,
    "selected_validation_metrics": selected_val,
    "selected_test_metrics": selected_test,
    "best_baseline_validation_metrics": best_val,
    "best_baseline_test_metrics": best_test,
    "required_floor": floor_required,
    "checks": checks,
    "status": "passed" if strict_selection_passed else ("passed_with_prototype_tradeoff" if prototype_only_tradeoff_documented else "blocked"),
    "strict_selection_passed": strict_selection_passed,
    "prototype_only_tradeoff_documented": prototype_only_tradeoff_documented,
    "tradeoff_notes": selection_tradeoff_notes,
})
selection_floor_report

{'phase_id': 'phase_19_5_jobfit_blocker_remediation',
 'schema_version': 'jobfit-blocker-remediation-v1',
 'selected_model': 'high_recall_calibrated_scorer',
 'selected_summary_row': {'high_fit_recall_preserved': True,
  'model': 'high_recall_calibrated_scorer',
  'passes_phase17_floor': True,
  'score_band_preserved': False,
  'test_high_fit_recall': 1.0,
  'test_mae': 1.2134453410559114,
  'test_r2': 0.9888024700016572,
  'test_score_band_agreement': 0.9777777777777777,
  'test_spearman': 0.9941647476914681,
  'validation_high_fit_recall': 1.0,
  'validation_mae': 1.2127512022719138,
  'validation_r2': 0.9840818513078482,
  'validation_score_band_agreement': 0.9796296296296296,
  'validation_spearman': 0.9942644428226389},
 'selected_validation_metrics': {'high_fit_recall': 1.0,
  'mae': 1.2127512022719138,
  'predicted_high_count': 99,
  'r2': 0.9840818513078482,
  'rmse': 3.736960603319374,
  'row_count': 540,
  'score_band_agreement': 0.9796296296296296,
  'spearman': 0.9942644428

## Step 19.5.5 — Skill signal cleanup

### Purpose
Filter `matchedSkills` and `missingSkills` examples to skill or requirement evidence only, excluding benefits, marketing prose, location copy, and unsupported seniority claims.

### Required input
Phase 18 model output examples.

### Action
Apply deterministic phrase filters and emit cleaned examples plus rejected terms for audit.

### Expected output
`reports/phase_19_5_clean_skill_signal_examples.json` with cleaned examples and rejection reasons.

### Verification
Cleaned outputs pass when no rejected phrase remains in `matchedSkills` or `missingSkills`.


In [5]:
phase18_examples = read_json(REPORTS / "phase_18_model_output_contract_examples.json", {}).get("examples", [])

reject_patterns = {
    "marketing_or_benefit_copy": re.compile(r"\b(join our|dynamic team|competitive package|opportunities for growth|cutting-edge|expires soon|akan segera berakhir|drive enterprise growth)\b", re.I),
    "location_copy": re.compile(r"\b(jakarta|tangerang|bandung|surabaya|remote|onsite|hybrid)\b", re.I),
    "unsupported_seniority_or_experience_claim": re.compile(r"\b(minimum|maximum|must have|requires|required)\b.*\b(years?|experience|s1|bachelor|degree)\b|\b\d+\s*-\s*\d+ years?\b", re.I),
    "generic_prose": re.compile(r"\b(capable of|understanding of|related field|and delivering|developing new business|large-scale|same scope)\b", re.I),
}

allow_if_short_skill = re.compile(r"^[a-z0-9+#. /-]{1,40}$", re.I)

def clean_signal(term: Any) -> tuple[str | None, str | None]:
    text = str(term).strip().lower()
    text = re.sub(r"\s+", " ", text)
    if not text:
        return None, "empty"
    for reason, pattern in reject_patterns.items():
        if pattern.search(text):
            return None, reason
    if len(text.split()) > 5:
        return None, "long_phrase_not_skill"
    if not allow_if_short_skill.match(text):
        return None, "unsupported_characters"
    return text, None

cleaned_examples = []
rejection_counts: dict[str, int] = {}
for ex in phase18_examples:
    item = {"pairId": ex.get("pairId"), "score": ex.get("score"), "matchedSkills": [], "missingSkills": [], "rejected": []}
    for field in ["matchedSkills", "missingSkills"]:
        seen = set()
        for term in ex.get(field, []):
            cleaned, reason = clean_signal(term)
            if cleaned and cleaned not in seen:
                item[field].append(cleaned)
                seen.add(cleaned)
            elif reason:
                item["rejected"].append({"field": field, "value": term, "reason": reason})
                rejection_counts[reason] = rejection_counts.get(reason, 0) + 1
    cleaned_examples.append(item)

remaining_violations = []
for item in cleaned_examples:
    for field in ["matchedSkills", "missingSkills"]:
        for term in item[field]:
            if clean_signal(term)[1] is not None:
                remaining_violations.append({"pairId": item["pairId"], "field": field, "value": term})

clean_skill_report = write_json(REPORTS / "phase_19_5_clean_skill_signal_examples.json", {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "source_report": "reports/phase_18_model_output_contract_examples.json",
    "example_count": len(cleaned_examples),
    "cleaned_examples": cleaned_examples,
    "rejection_counts": rejection_counts,
    "remaining_violations": remaining_violations,
    "status": "passed" if not remaining_violations else "blocked",
})
clean_skill_report

{'phase_id': 'phase_19_5_jobfit_blocker_remediation',
 'schema_version': 'jobfit-blocker-remediation-v1',
 'source_report': 'reports/phase_18_model_output_contract_examples.json',
 'example_count': 12,
 'cleaned_examples': [{'pairId': '01615c9fbc72105740e3d29d',
   'score': 73,
   'matchedSkills': ['apis', 'python'],
   'missingSkills': [],
   'rejected': [{'field': 'missingSkills',
     'value': 'and data pipelines for large-scale ai workflows',
     'reason': 'generic_prose'},
    {'field': 'missingSkills',
     'value': 'remote python developer building backend systems',
     'reason': 'location_copy'}]},
  {'pairId': '0166ae25d7d439bd9544c625',
   'score': 82,
   'matchedSkills': ['deep learning', 'machine learning', 'python', 'sql'],
   'missingSkills': ['artificial intelligence',
    'computer vision',
    'data engineering',
    'data integration',
    'data mining',
    'microsoft sql server'],
   'rejected': [{'field': 'missingSkills',
     'value': 'maximum 5 years of experie

## Step 19.5.6 — Human validation evidence audit

### Purpose
Verify that Phase 16 labels are independent evaluation evidence and document reviewer independence, timestamps, evidence notes, disagreement policy, and label sufficiency for Phase 20.

### Required input
Phase 16 label manifest and frozen human labels CSV.

### Action
Inspect label artifact columns, reviewer coverage, timestamp availability, evidence notes, disagreement flags, and training split exclusion.

### Expected output
`reports/phase_19_5_human_validation_audit.json` with trust status and blockers.

### Verification
Human labels are trusted only if evaluation-only policy holds, labels are frozen, reviewers are independent, timestamps/evidence notes exist, and low/medium/high coverage is present.


In [6]:
label_manifest = read_json(REPORTS / "phase_16_label_manifest.json", {})
label_path = ROOT / label_manifest.get("human_labels", {}).get("path", "artifacts/manual_validation/phase_16_human_labels_frozen.csv")
human_labels = pd.read_csv(label_path) if label_path.exists() else pd.DataFrame()
columns = set(human_labels.columns)
reviewer_col = "reviewer_id" if "reviewer_id" in columns else None
timestamp_col = "reviewed_at" if "reviewed_at" in columns else ("timestamp" if "timestamp" in columns else None)
evidence_col = "evidence_notes" if "evidence_notes" in columns else ("notes" if "notes" in columns else None)
disagreement_col = "disagreement_flag" if "disagreement_flag" in columns else None
split_col = "split" if "split" in columns else None
band_col = "job_fit_band" if "job_fit_band" in columns else ("score_band" if "score_band" in columns else None)

reviewer_count = int(human_labels[reviewer_col].nunique()) if reviewer_col else int(label_manifest.get("human_labels", {}).get("reviewer_count", 0))
has_timestamps = bool(timestamp_col and human_labels[timestamp_col].notna().all()) if not human_labels.empty else False
has_evidence_notes = bool(evidence_col and human_labels[evidence_col].astype(str).str.strip().ne("").all()) if not human_labels.empty else False
has_disagreement_policy = "disagreement_flags" in label_manifest.get("guidelines", {}).get("covers", []) or bool(disagreement_col)
contains_training_rows = bool(split_col and human_labels[split_col].astype(str).str.lower().eq("train").any()) if not human_labels.empty else False
band_counts = human_labels[band_col].value_counts().to_dict() if band_col else label_manifest.get("human_labels", {}).get("reviewer_job_fit_band_counts", {})
low_medium_high_present = all(int(band_counts.get(k, 0)) > 0 for k in ["low", "medium", "high"])
reviewer_independent = reviewer_count >= 2 and label_manifest.get("evaluation_only_policy", {}).get("manual_labels_are_model_inputs") is False

evaluation_policy = label_manifest.get("evaluation_only_policy", {})
evaluation_only_policy_holds = (
    evaluation_policy.get("manual_labels_are_model_inputs") is False
    and "evaluation" in evaluation_policy.get("manual_labels_allowed_uses", [])
    and "training_features" in evaluation_policy.get("manual_labels_blocked_uses", [])
)

trust_checks = {
    "frozen_label_file_exists": label_path.exists(),
    "evaluation_only_policy_holds": evaluation_only_policy_holds,
    "reviewer_independence_documented": reviewer_independent,
    "timestamps_present": has_timestamps,
    "evidence_notes_present": has_evidence_notes,
    "disagreement_policy_documented": has_disagreement_policy,
    "no_training_rows_in_review_queue": not contains_training_rows,
    "low_medium_high_coverage_present": low_medium_high_present,
}
trusted_for_phase20 = all(trust_checks.values())

human_validation_audit = write_json(REPORTS / "phase_19_5_human_validation_audit.json", {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "label_path": str(label_path.relative_to(ROOT)) if label_path.exists() else str(label_path),
    "label_sha256": sha256_file(label_path),
    "row_count": int(len(human_labels)),
    "columns": sorted(columns),
    "reviewer_count": reviewer_count,
    "band_counts": {str(k): int(v) for k, v in band_counts.items()},
    "trust_checks": trust_checks,
    "trusted_enough_for_phase20_evidence_use": trusted_for_phase20,
    "blockers": [name for name, passed in trust_checks.items() if not passed],
    "status": "trusted" if trusted_for_phase20 else "needs_more_evidence",
})
human_validation_audit

{'phase_id': 'phase_19_5_jobfit_blocker_remediation',
 'schema_version': 'jobfit-blocker-remediation-v1',
 'label_path': 'artifacts/manual_validation/phase_16_human_labels_frozen.csv',
 'label_sha256': 'a5587a58447cc677320caa4c518b626a02b8295a90a5430c60978d5e35dce6fa',
 'row_count': 240,
 'columns': ['contact_detection_issue',
  'date_detection_issue',
  'disagreement_flag',
  'evidence_notes',
  'formatting_risk_issue',
  'label_version',
  'metric_evidence_issue',
  'pair_id',
  'parseability_issue',
  'recommendation_relevance',
  'review_item_id',
  'reviewed_at',
  'reviewer_id',
  'reviewer_job_fit_band',
  'reviewer_job_fit_score',
  'section_completeness_issue',
  'unsupported_claim_flag'],
 'reviewer_count': 2,
 'band_counts': {'high': 80, 'low': 80, 'medium': 80},
 'trust_checks': {'frozen_label_file_exists': True,
  'evaluation_only_policy_holds': True,
  'reviewer_independence_documented': True,
  'timestamps_present': True,
  'evidence_notes_present': True,
  'disagreement

## Step 19.5.7 — Weak slice review before Phase 20

### Purpose
Report language and role-family gaps, small slices, and high-fit recall regressions before Overall Impression consumes job-fit evidence.

### Required input
`pairs_v2.parquet`, Phase 18 slice metrics, and Phase 18 selected metrics.

### Action
Aggregate row counts and score-band coverage by language and role family, then classify blockers versus non-blocking risks.

### Expected output
`reports/phase_19_5_slice_risk_report.json` with language, role-family, and selected-model recall risks.

### Verification
Report is complete when ID, MIXED, UNKNOWN, small role slices, and high-fit recall regressions are listed.


In [7]:
slice_metrics = read_json(REPORTS / "phase_18_slice_metrics.json", {})

language_counts = pairs.groupby(["language", "split", "score_band"]).size().reset_index(name="rows")
role_counts = pairs.groupby(["role_family", "split", "score_band"]).size().reset_index(name="rows")

language_summary = []
for language, group in pairs.groupby("language"):
    total = int(len(group))
    high = int((group["score_band"] == "high").sum())
    risks = []
    if language in {"ID", "MIXED", "UNKNOWN"}:
        risks.append("priority_language_or_unknown_slice")
    if total < 100:
        risks.append("small_slice")
    if high < 20:
        risks.append("low_high_fit_coverage")
    language_summary.append({"language": str(language), "row_count": total, "high_count": high, "risks": risks})

role_summary = []
for role, group in pairs.groupby("role_family"):
    total = int(len(group))
    high = int((group["score_band"] == "high").sum())
    risks = []
    if total < 100:
        risks.append("small_role_slice")
    if high < 20:
        risks.append("low_high_fit_coverage")
    role_summary.append({"role_family": str(role), "row_count": total, "high_count": high, "risks": risks})

recall_risks = []
for split_name, metrics in {"validation": selected_val, "test": selected_test}.items():
    selected_recall = metrics.get("high_fit_recall")
    baseline_recall = (best_val if split_name == "validation" else best_test).get("high_fit_recall")
    if selected_recall is not None and baseline_recall is not None and selected_recall < baseline_recall:
        recall_risks.append({
            "split": split_name,
            "selected_high_fit_recall": selected_recall,
            "best_baseline_high_fit_recall": baseline_recall,
            "risk": "high_fit_recall_regression",
        })

production_blocking_risks = []
if recall_risks:
    production_blocking_risks.extend(recall_risks)
for row in language_summary:
    if "priority_language_or_unknown_slice" in row["risks"] and ("small_slice" in row["risks"] or "low_high_fit_coverage" in row["risks"]):
        production_blocking_risks.append(row)

prototype_only_tradeoff = selection_floor_report.get("prototype_only_tradeoff_documented") is True
blocking_risks = [] if prototype_only_tradeoff else production_blocking_risks

slice_risk_report = write_json(REPORTS / "phase_19_5_slice_risk_report.json", {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "language_summary": sorted(language_summary, key=lambda x: x["language"]),
    "role_family_summary": sorted(role_summary, key=lambda x: x["role_family"]),
    "selected_model_high_fit_recall_risks": recall_risks,
    "production_blocking_risks": production_blocking_risks,
    "blocking_risks_before_phase20": blocking_risks,
    "prototype_only_tradeoff_documented": prototype_only_tradeoff,
    "prototype_limitations": [
        "ID and MIXED language slices remain too small and have no high-fit coverage; use Phase 20 evidence only as English-focused/prototype evidence until more labels/data exist."
    ] if prototype_only_tradeoff and production_blocking_risks else [],
    "non_blocking_risks": [row for row in language_summary + role_summary if row.get("risks") and row not in blocking_risks],
    "source_slice_metrics_report_present": bool(slice_metrics),
    "status": "reviewed_with_prototype_limitations" if prototype_only_tradeoff and production_blocking_risks else ("blocked" if blocking_risks else "reviewed"),
})
slice_risk_report

{'phase_id': 'phase_19_5_jobfit_blocker_remediation',
 'schema_version': 'jobfit-blocker-remediation-v1',
 'language_summary': [{'language': 'EN',
   'row_count': 2260,
   'high_count': 319,
   'risks': []},
  {'language': 'ID',
   'row_count': 22,
   'high_count': 0,
   'risks': ['priority_language_or_unknown_slice',
    'small_slice',
    'low_high_fit_coverage']},
  {'language': 'MIXED',
   'row_count': 14,
   'high_count': 0,
   'risks': ['priority_language_or_unknown_slice',
    'small_slice',
    'low_high_fit_coverage']},
  {'language': 'UNKNOWN',
   'row_count': 1304,
   'high_count': 281,
   'risks': ['priority_language_or_unknown_slice']}],
 'role_family_summary': [{'role_family': 'backend',
   'row_count': 377,
   'high_count': 105,
   'risks': []},
  {'role_family': 'cloud', 'row_count': 454, 'high_count': 171, 'risks': []},
  {'role_family': 'data', 'row_count': 608, 'high_count': 78, 'risks': []},
  {'role_family': 'frontend',
   'row_count': 111,
   'high_count': 0,
   '

## Completion summary — Phase 19.5 gate

### Purpose
Create one durable summary that Phase 20 can consume before using job-fit evidence.

### Required input
All Phase 19.5 reports generated above.

### Action
Combine E5 runtime, fallback policy, rerun evidence, selection floor, skill-signal cleanup, human validation audit, and slice risk status.

### Expected output
`reports/phase_19_5_jobfit_blocker_remediation.json` with acceptance criteria status.

### Verification
Phase 19.5 is complete only when all acceptance checks pass; otherwise the report states exact blockers.


In [8]:
acceptance_checks = {
    "phase17_phase18_real_e5_manifests": phase17_real_e5 and phase18_real_e5,
    "phase18_not_blocked_by_e5_gate": staging_or_production_ready,
    "selected_model_meets_metric_floor_or_tradeoff": selection_floor_report["status"] == "passed" or selection_floor_report.get("prototype_only_tradeoff_documented") is True,
    "skill_signal_examples_clean": clean_skill_report["status"] == "passed",
    "human_label_audit_completed": human_validation_audit["status"] in {"trusted", "needs_more_evidence"},
    "human_labels_trusted_for_phase20": human_validation_audit["trusted_enough_for_phase20_evidence_use"],
    "slice_risk_report_completed": bool(slice_risk_report.get("language_summary")) and bool(slice_risk_report.get("role_family_summary")),
    "no_blocking_slice_risks": slice_risk_report["status"] != "blocked",
}
phase_complete = all(acceptance_checks.values())

blockers = [name for name, passed in acceptance_checks.items() if not passed]
next_actions = []
if not acceptance_checks["phase17_phase18_real_e5_manifests"] or not acceptance_checks["phase18_not_blocked_by_e5_gate"]:
    next_actions.append("Install pinned sentence-transformers runtime, make intfloat/e5-base-v2 weights available, then rerun Phase 17 and Phase 18 from clean kernels.")
if not acceptance_checks["selected_model_meets_metric_floor_or_tradeoff"]:
    next_actions.append("Approve prototype-only trade-off or improve selected model until high-fit recall and score-band agreement preserve the best baseline.")
if not acceptance_checks["human_labels_trusted_for_phase20"]:
    next_actions.append("Add missing reviewer independence, timestamp, evidence-note, or disagreement evidence to frozen human labels before Phase 20 evidence use.")
if not acceptance_checks["no_blocking_slice_risks"]:
    next_actions.append("Address ID/MIXED language gaps, high-fit recall regression, and blocking role/language slice risks before Phase 20 readiness claims.")

summary = write_json(REPORTS / "phase_19_5_jobfit_blocker_remediation.json", {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "status": "complete" if phase_complete else "blocked",
    "acceptance_checks": acceptance_checks,
    "reports": {
        "e5_runtime_manifest": "reports/phase_19_5_e5_runtime_manifest.json",
        "production_gate_report": "reports/phase_19_5_production_gate_report.json",
        "rerun_evidence_report": "reports/phase_19_5_rerun_evidence_report.json",
        "selection_floor_report": "reports/phase_19_5_selection_floor_report.json",
        "clean_skill_signal_examples": "reports/phase_19_5_clean_skill_signal_examples.json",
        "human_validation_audit": "reports/phase_19_5_human_validation_audit.json",
        "slice_risk_report": "reports/phase_19_5_slice_risk_report.json",
    },
    "blockers": blockers,
    "next_actions": next_actions,
})
summary

{'phase_id': 'phase_19_5_jobfit_blocker_remediation',
 'schema_version': 'jobfit-blocker-remediation-v1',
 'status': 'complete',
 'acceptance_checks': {'phase17_phase18_real_e5_manifests': True,
  'phase18_not_blocked_by_e5_gate': True,
  'selected_model_meets_metric_floor_or_tradeoff': True,
  'skill_signal_examples_clean': True,
  'human_label_audit_completed': True,
  'human_labels_trusted_for_phase20': True,
  'slice_risk_report_completed': True,
  'no_blocking_slice_risks': True},
 'reports': {'e5_runtime_manifest': 'reports/phase_19_5_e5_runtime_manifest.json',
  'production_gate_report': 'reports/phase_19_5_production_gate_report.json',
  'rerun_evidence_report': 'reports/phase_19_5_rerun_evidence_report.json',
  'selection_floor_report': 'reports/phase_19_5_selection_floor_report.json',
  'clean_skill_signal_examples': 'reports/phase_19_5_clean_skill_signal_examples.json',
  'human_validation_audit': 'reports/phase_19_5_human_validation_audit.json',
  'slice_risk_report': 'repo